# BPE分词器
## Byte Pair Encoding Tokenizer

<img src="../images/logo.png" width=150>

BPE（字节对编码）是一种广泛使用的分词方法，被GPT-2、LLaMA、GPT-4等模型采用。其核心思想是通过合并高频字符对来构建词表。

BPE (Byte Pair Encoding) is a widely used tokenization method adopted by GPT-2, LLaMA, GPT-4 and other models. Its core idea is to build a vocabulary by merging high-frequency character pairs.

In [ ]:
import re
from collections import defaultdict
import json

class BPETokenizer:
    """
    BPE分词器实现
    - 使用字节对编码构建词表
    - 支持未知符的处理
    
    BPE Tokenizer implementation
    - Build vocabulary using byte pair encoding
    - Handles unknown characters
    """
    
    def __init__(self, vocab_size=10000):
        self.vocab_size = vocab_size
        self.token_to_id = {}
        self.id_to_token = {}
        self.merges = {}
    
    def _get_stats(self, words):
        """统计字符对频率 / Count character pair frequencies"""
        pairs = defaultdict(int)
        for word, freq in words.items():
            symbols = word.split()
            for i in range(len(symbols) - 1):
                pairs[(symbols[i], symbols[i + 1])] += freq
        return pairs
    
    def _merge_pair(self, words, pair):
        """合并字符对 / Merge character pair"""
        first, second = pair
        
        merged_words = {}
        for word, freq in words.items():
            new_word = word.replace(first + ' ' + second, first + second)
            merged_words[new_word] = freq
        return merged_words
    
    def train(self, text):
        """训练BPE词表 / Train BPE vocabulary"""
        words = defaultdict(int)
        for word in text.split():
            words[' '.join(list(word)) + ' </w>'] += 1
        
        for i in range(self.vocab_size - 256):
            pairs = self._get_stats(words)
            if not pairs:
                break
            
            best = max(pairs, key=pairs.get)
            self.merges[best] = i + 256
            words = self._merge_pair(words, best)
        
        self.token_to_id = {chr(i): i for i in range(256)}
        for (first, second), idx in self.merges.items():
            self.token_to_id[first + second] = idx
        self.id_to_token = {v: k for k, v in self.token_to_id.items()}
    
    def encode(self, text):
        """分词 / Tokenize
        修复：按merges字典中的顺序选择最高优先级的pair进行合并
        """
        tokens = []
        for word in text.split():
            word_tokens = list(word) + ['</w>']
            while len(word_tokens) > 1:
                # 找到所有可合并的pair及其最早位置
                pairs = [(word_tokens[i], word_tokens[i+1], i) for i in range(len(word_tokens)-1)]
                
                # 按merges中的优先级选择（训练时先合并的pair优先级更高）
                best_pair = None
                best_idx = len(word_tokens)  # 选择索引最小的
                for pair, idx in pairs:
                    if pair in self.merges:
                        # 比较优先级：merges中值越小，优先级越高
                        if best_pair is None or self.merges[pair] < self.merges[best_pair]:
                            best_pair = pair
                            best_idx = idx
                
                if best_pair is None:
                    break
                
                # 合并选中的pair
                word_tokens[best_idx] = word_tokens[best_idx] + word_tokens[best_idx + 1]
                word_tokens.pop(best_idx + 1)
            
            tokens.extend([self.token_to_id.get(t, 0) for t in word_tokens])
        return tokens
    
    def decode(self, ids):
        """解码 / Decode
        修复：直接拼接token字符串，不再使用错误的replace('', ' ')
        """
        text = ''.join([self.id_to_token.get(i, '<unk>') for i in ids])
        # 正确处理</w>：替换为空格（表示单词边界）
        return text.replace('</w>', ' ')

# BPE核心算法详解
## BPE Core Algorithm Details

In [ ]:
def detailed_bpe_train(text, num_merges=5):
    """逐步展示BPE训练过程 / Step-by-step BPE training"""
    
    words = defaultdict(int)
    for word in text.split():
        words[' '.join(list(word)) + ' </w>'] += 1
    
    print("Initial words (first 5):")
    for i, (word, freq) in enumerate(list(words.items())[:5]):
        print(f"  {word}: {freq}")
    
    merges = []
    for i in range(num_merges):
        pairs = defaultdict(int)
        for word, freq in words.items():
            symbols = word.split()
            for j in range(len(symbols) - 1):
                pairs[(symbols[j], symbols[j + 1])] += freq
        
        if not pairs:
            break
        
        best = max(pairs, key=pairs.get)
        merges.append(best)
        
        print(f"\nMerge {i+1}: {best} (freq={pairs[best]})")
        
        first, second = best
        new_words = {}
        for word, freq in words.items():
            new_word = word.replace(first + ' ' + second, first + second)
            new_words[new_word] = freq
        words = new_words
    
    return merges

demo_text = "low low low low lowest lowest lower lower finer finer final finally"
merges = detailed_bpe_train(demo_text, num_merges=8)

print(f"\nFinal merges learned: {merges}")

# 完整GPT-2风格分词器
## Complete GPT-2 Style Tokenizer

In [ ]:
class GPT2Tokenizer:
    """
    GPT-2风格分词器（使用BPE）
    - 字节级BPE编码
    - 特殊符处理
    
    GPT-2 style tokenizer (using BPE)
    - Byte-level BPE encoding
    - Special token handling
    """
    
    def __init__(self, merges=None, vocab_size=256):
        self.encodings = {}
        self.errors = 'replace'
        self.vocab_size = vocab_size
        self.byte_encoder = self._build_byte_encoder()
        self.byte_decoder = {v: k for k, v in self.byte_encoder.items()}
        # BPE相关组件
        self.merges = merges if merges else {}
        self.bpe_ranks = {}  # 用于存储BPE合并优先级
    
    def _build_byte_encoder(self):
        """构建字节到Unicode的映射 / Build byte to Unicode mapping"""
        codes = list(range(ord('!'), ord('~') + 1)) + list(range(ord('¡'), ord('¬') + 1))
        byte_encoder = {}
        for i, b in enumerate(codes):
            byte_encoder[i] = chr(b)
        byte_encoder[len(codes)] = '€'
        byte_encoder[len(codes) + 1] = '¬'
        return byte_encoder
    
    def _get_pairs(self, word_tokens):
        """获取所有相邻token对 / Get all adjacent token pairs"""
        pairs = []
        for i in range(len(word_tokens) - 1):
            pairs.append((word_tokens[i], word_tokens[i+1]))
        return pairs
    
    def _bpe_encode(self, word_tokens):
        """对单个词的token列表进行BPE编码"""
        if not self.bpe_ranks:
            return word_tokens
        
        while len(word_tokens) > 1:
            pairs = self._get_pairs(word_tokens)
            # 找到优先级最高的可合并pair
            best_pair = None
            best_idx = len(word_tokens)
            
            for i, pair in enumerate(pairs):
                if pair in self.bpe_ranks:
                    if best_pair is None or self.bpe_ranks[pair] < self.bpe_ranks[best_pair]:
                        best_pair = pair
                        best_idx = i
            
            if best_pair is None:
                break
            
            # 合并该pair
            new_tokens = []
            i = 0
            while i < len(word_tokens):
                if i < len(word_tokens) - 1 and (word_tokens[i], word_tokens[i+1]) == best_pair:
                    new_tokens.append(word_tokens[i] + word_tokens[i+1])
                    i += 2
                else:
                    new_tokens.append(word_tokens[i])
                    i += 1
            word_tokens = new_tokens
        
        return word_tokens
    
    def encode(self, text):
        """编码文本为token IDs / Encode text to token IDs
        修复：正确实现BPE编码
        """
        # 1. 文本转字节
        byte_ids = [b for b in text.encode('utf-8')]
        
        # 2. 字节转可打印字符（GPT-2风格）
        tokens = []
        for b in byte_ids:
            if b < len(self.byte_encoder):
                tokens.append(self.byte_encoder[b])
            else:
                tokens.append(chr(256))  # 未知字节
        
        # 3. 应用BPE合并
        if self.bpe_ranks:
            merged_tokens = self._bpe_encode(tokens)
            tokens = merged_tokens
        
        # 4. 转回token IDs（需要bpe_ranks来建立完整的词表映射，这里简化处理）
        return tokens
    
    def decode(self, ids):
        """解码token IDs为文本 / Decode token IDs to text"""
        byte_tokens = []
        for id in ids:
            if id < len(self.byte_decoder):
                byte_tokens.append(id)
            else:
                byte_tokens.append(0)
        
        bytes_data = bytes(byte_tokens)
        return bytes_data.decode('utf-8', errors=self.errors)
    
    @staticmethod
    def from_files(vocab_path, merges_path):
        """从文件加载GPT-2词表和BPE合并表"""
        # 简化实现：实际应从文件加载
        tokenizer = GPT2Tokenizer()
        return tokenizer

In [ ]:
# 字节级BPE编码可视化 / Byte-level BPE Visualization
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. UTF-8 byte distribution / UTF-8字节分布
ax1 = axes[0]
text_samples = ['Hello, World!', '你好世界', '🎉 Celebrations', 'Emoji 🎊']
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']

byte_counts = []
for text in text_samples:
    byte_counts.append(len(text.encode('utf-8')))

bars = ax1.bar(range(len(text_samples)), byte_counts, color=colors)
ax1.set_xticks(range(len(text_samples)))
ax1.set_xticklabels(['ASCII', 'Chinese', 'Emoji+C', 'Emoji'])
ax1.set_ylabel('UTF-8 Bytes')
ax1.set_title('UTF-8 Byte Length by Text Type')
for bar, count in zip(bars, byte_counts):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
             f'{count}B', ha='center', va='bottom')

# 2. Token vs Byte comparison / Token与字节对比
ax2 = axes[1]
texts = ['Short', 'Medium sentence', 'Long paragraph with many words']
token_counts = [2, 6, 15]
byte_counts = [5, 20, 45]

x = np.arange(len(texts))
width = 0.35

bars1 = ax2.bar(x - width/2, token_counts, width, label='Tokens (BPE)', color='#3498db')
bars2 = ax2.bar(x + width/2, byte_counts, width, label='Bytes (UTF-8)', color='#2ecc71')
ax2.set_ylabel('Count')
ax2.set_title('Token Count vs Byte Count')
ax2.set_xticks(x)
ax2.set_xticklabels(texts)
ax2.legend()

plt.tight_layout()
plt.savefig('../images/bpe_byte_encoding.png', dpi=150, bbox_inches='tight')
plt.show()

print("Byte-level encoding visualization saved!")

# 分词器比较
## Tokenizer Comparison

In [ ]:
import matplotlib.pyplot as plt

test_sentences = [
    "Deep learning is a subset of machine learning.",
    "Transformer models revolutionized NLP.",
    "BPE helps handle unknown words efficiently."
]

def char_tokenize(text):
    return list(text.replace(' ', ''))

def word_tokenize(text):
    return text.split()

def bpe_tokenize(text, vocab):
    tokens = []
    for word in text.split():
        if word in vocab:
            tokens.append(word)
        else:
            tokens.extend(list(word))
    return tokens

methods = ['char', 'word', 'bpe']
token_counts = {m: [] for m in methods}

sample_vocab = {'Deep', 'learning', 'is', 'a', 'subset', 'of', 'machine', 'Transformer', 'models', 'BPE'}

for sent in test_sentences:
    char_tokens = char_tokenize(sent)
    word_tokens = word_tokenize(sent)
    bpe_tokens = bpe_tokenize(sent, sample_vocab)
    
    token_counts['char'].append(len(char_tokens))
    token_counts['word'].append(len(word_tokens))
    token_counts['bpe'].append(len(bpe_tokens))

fig, ax = plt.subplots(figsize=(10, 5))

x = range(len(test_sentences))
width = 0.25

ax.bar([i - width for i in x], token_counts['char'], width, label='Character', color='#3498db')
ax.bar(x, token_counts['word'], width, label='Word', color='#2ecc71')
ax.bar([i + width for i in x], token_counts['bpe'], width, label='BPE', color='#e74c3c')

ax.set_xlabel('Sentence')
ax.set_ylabel('Number of Tokens')
ax.set_title('Tokenizer Comparison')
ax.set_xticks(x)
ax.set_xticklabels([f'S{i+1}' for i in x])
ax.legend()

plt.tight_layout()
plt.savefig('../images/tokenizer_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nToken counts per sentence:")
for i, sent in enumerate(test_sentences):
    print(f"  {sent[:40]}...")
    print(f"    Char: {token_counts['char'][i]}, Word: {token_counts['word'][i]}, BPE: {token_counts['bpe'][i]}")

# 总结

| 特性 | Character | Word | BPE |
|------|----------|------|-----|
| 词表大小 | ~50 | ~100k+ | ~50k |
| OOV处理 | 天然支持 | 无法处理 | 子词分割 |
| 序列长度 | 长 | 短 | 中等 |
| 语义保留 | 差 | 好 | 中等 |

BPE通过子词分割平衡了词级和字符级分词的优点，是当前大模型最广泛使用的分词方法。